# 9.10 信心和正確率對得上嗎？

# 第 9 章：誠實、安全與適當回應

前置：第7章SFT與第8章分項評估。用能驗證真假的小世界教誠實、修正前提與安全邊界，不把課堂成功推廣成任意開放世界安全保證。外部PKU資料的非商用授權與原始安全標籤要保留。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：信心像音量：說得大聲不保證說對**

每箱保留樣本量；文字自述信心是另一回事。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/calibration.svg")))

**先想一想：**信心0.9的題目只有一半答對，曲線在哪裡？

把預測按信心分箱，比每箱平均信心與正確率。理想校準在對角線附近；每箱樣本太少時曲線會很不穩。

**把直覺寫成數學：**每箱accuracy=correct_count/count；ECE依箱樣本量加權|accuracy−confidence|。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.alignment import reliability_bins

confidence = torch.tensor([0.55, 0.65, 0.85, 0.95])
correct = torch.tensor([1, 0, 1, 0])
bins = reliability_bins(confidence, correct, bins=2)
print(bins)
import matplotlib.pyplot as plt

plt.plot([0, 1], [0, 1], "--", label="理想校準")
plt.scatter([b["confidence"] for b in bins], [b["accuracy"] for b in bins])
plt.xlabel("mean confidence")
plt.ylabel("accuracy")
plt.show()

**如何讀結果：**四筆只為展示計算；真校準應另用validation調溫度，在test報告。

**只改一件事：**只增加低信心但正確的例子，核對箱樣本量。
